# RODA · Evaluación externa semanal de demanda

Este cuaderno ejecuta cinco modelos y cuatro referencias sobre **Sales Transactions Weekly** de UCI (811 productos anónimos, 52 semanas, DOI: [10.24432/C5XS4Q](https://doi.org/10.24432/C5XS4Q), licencia CC BY 4.0). No son repuestos identificados de RODA ni ventas peruanas. La fuente original y su hash están en `ml/examples/uci-weekly/source.zip`.

**Protocolo fijado:** entrenar con objetivos W8–W35, elegir modelos y suavizados con W36–W43 y reservar W44–W51 para una prueba final. Cada pronóstico usa solo las ventas anteriores a su semana. Después de elegir el método, los cinco modelos se reajustan hasta W43; la prueba no interviene en la selección.


In [ ]:
from pathlib import Path
import os, subprocess, sys
if not Path('ml/weekly_external.py').exists():
    subprocess.run(['git','clone','https://github.com/NEYEN111/REPUESTOS.git'],check=True)
    os.chdir('REPUESTOS')
subprocess.run([sys.executable,'-m','pip','-q','install','-r','ml/requirements.txt'],check=True)
print('Proyecto:',Path.cwd())


## Ejecutar la evaluación
El script valida el SHA-256 de la fuente, excluye columnas derivadas de la serie completa y guarda métricas, predicciones por producto y gráficos. Vuelve a correrlo para repetir el experimento.


In [ ]:
import json, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable,'-m','ml.weekly_external'],check=True)
report=json.loads(Path('ml/evaluations/uci-weekly-v1/report.json').read_text(encoding='utf-8'))
print('Fuente:',report['source']['title'],'·',report['source']['doi'])
print('Separación:',report['protocol']['train_targets'],report['protocol']['validation_targets'],report['protocol']['test_targets'])
print('Filas:',report['data']['training_rows'],report['data']['validation_rows'],report['data']['test_rows'])


## Cinco modelos y referencias
La primera tabla sirve para elegir el método. La segunda es la prueba reservada. MAE representa error absoluto medio en unidades por producto y semana; menor es mejor. El WAPE puede ser alto en productos de escasa venta y no debe interpretarse como porcentaje de exactitud.


In [ ]:
import pandas as pd
from IPython.display import display
root=Path('ml/evaluations/uci-weekly-v1')
selection=pd.read_csv(root/'validation_methods.csv')
test=pd.read_csv(root/'test_methods.csv')
display(selection[['method','family','mae','wape_pct']].round(3))
display(test[['method','family','mae','wape_pct']].round(3))
print('ML elegido en validación:',report['selection']['ml_winner'])
print('Referencia elegida en validación:',report['selection']['reference_winner'])
print('Decisión final:',report['test']['decision'])


In [ ]:
from IPython.display import Image, display
for name in ('01_metodos.png','02_rotacion.png'):
    display(Image(filename=str(root/'images'/name)))
display(pd.read_csv(root/'test_by_rotation.csv').round(3))


## Trazabilidad y uso responsable
`test_predictions.csv` conserva las 6 488 observaciones reservadas, el valor real y las estimaciones congeladas. `selected_model.joblib` permite reproducir el pronóstico del ML elegido, pero **no es un modelo operativo de RODA**. El experimento compara métodos semanales en otra tienda; tampoco contiene stock, demanda perdida ni plazos de proveedores. Si ningún modelo supera la referencia predefinida por al menos 1% en validación y prueba, se conserva la referencia y el modelo queda sin aprobar. La siguiente evaluación de RODA debe usar cierres diarios completos de ventas propias.


In [ ]:
pred=pd.read_csv(root/'test_predictions.csv')
assert len(pred)==report['data']['test_rows']==6488
assert pred.week.unique().tolist()==[f'W{i}' for i in range(44,52)]
assert report['source']['archive_sha256']=='dededa20f199af8f2bb96184bb1696befda902849a700bde283fe763afc90ba9'
display(pred.head(12))


In [ ]:
import hashlib, joblib, numpy as np
from ml.weekly_external import load_source, feature_matrix
artifact=root/report['selection']['selected_model_artifact']
assert hashlib.sha256(artifact.read_bytes()).hexdigest()==report['selection']['selected_model_sha256']
loaded=joblib.load(artifact)
ids,values,_=load_source()
week44,_=feature_matrix(values,range(44,45))
forecast=np.maximum(0,loaded.predict(week44))
assert np.allclose(forecast,np.array(pred.loc[pred.week=='W44','ml_selected']),atol=0.0001)
print('Modelo guardado y verificado:',report['selection']['ml_winner'])
display(pd.DataFrame({'producto':ids[:10],'pronóstico_W44':forecast[:10],'venta_W44':values[:10,44]}))
